# Step 07 — Rule engine on the validation set

**Input** — `data/validation/06_validation_set.csv` and the CURRENT run's

`05_condensed_buildings_with_pois.gpkg`

**Output** — `rule_predictions.parquet` + `rule_comparison.csv`

The rule engine reads **tags**, so it needs the building's actual tag columns.

Those are pulled from the current run of notebook 05 and joined on

`source_gml_id` — the register id, stable across runs. Joining on `gml_id` would

silently attach a different building's tags (see notebook 06).

`rule_utils.classify_building()` is fully deterministic: every answer comes from

an explicit lookup table, so this notebook produces the same numbers every time

it is run. That is the property notebook 10 shows the LLM does not have.

In [ ]:
import sys

sys.path.insert(0, str(__import__('pathlib').Path('..').resolve()))

import pandas as pd

import pyogrio

from config import (VALIDATION_SET_FILE, CONDENSED_BUILDINGS_FILE,

                    arm_predictions, arm_comparison)

from rule_utils import classify_building

from validation_utils import (decode_final_validation_set, collapse_to_zone_activities,

                              build_comparison)

ARM = 'rule'

pd.set_option('display.width', 200)

print('Config loaded')

## 1. Load the validation set and decode its ground truth

In [ ]:
val = decode_final_validation_set(pd.read_csv(VALIDATION_SET_FILE))

print(f'{len(val):,} validated buildings')

print(f"  activities scoreable : {val['activities_truth'].notna().sum():,}")

print(f"  bosserhof scoreable  : {val['bosserhof_truth'].notna().sum():,}")

## 2. Pull tag columns from the current run

One join on `source_gml_id`. `function` is the raw ALKIS building-function code

and is what `rule_utils` layer 2 is keyed on; the rest feed layers 1 and 3.

A few validated rows share a `source_gml_id` (notebook 05 merged buildings that

were annotated separately), so this is many-to-one by design — each annotation

gets the tags of the building it now belongs to. `validate='m:1'` states that:

it still fails loudly if the *right* side ever gains a duplicate, which would mean

the join key had stopped identifying a single building.

In [ ]:
TAG_COLS = ['function', 'osm_building_type', 'osm_landuse_class', 'amenity', 'shop',

            'tourism', 'building', 'information', 'additional_information']

tags = pyogrio.read_dataframe(CONDENSED_BUILDINGS_FILE,

                              columns=['source_gml_id'] + TAG_COLS,

                              read_geometry=False)

print(f'{len(tags):,} buildings in the current run')

work = val.merge(tags, on='source_gml_id', how='left', validate='m:1', indicator='_matched')

matched = work['_matched'] == 'both'

assert matched.all(), (f'{(~matched).sum()} validated rows found no building in the current '

                       'run — re-run notebook 06, its source_gml_id is stale')

work = work.drop(columns='_matched')

print(f'matched by source_gml_id : {matched.sum():,} / {len(work):,}')

has_signal = work[TAG_COLS].notna().any(axis=1)

assert has_signal.all(), f'{(~has_signal).sum()} rows have no tag data in the current run'

print('every validated row carries at least one tag column')

## 3. Classify

In [ ]:
preds = pd.DataFrame([classify_building(r) for _, r in work.iterrows()])

preds['gml_id'] = work['gml_id'].values

preds['pred_bosserhof'] = preds['bosserhof_class']

preds['pred_zone_activities'] = preds['mid_labels'].map(collapse_to_zone_activities)

preds.to_parquet(arm_predictions(ARM), index=False)

print(f'classified {len(preds):,} buildings -> {arm_predictions(ARM).name}')

print(f"\nrule layer that answered:\n{preds['interpreted_type'].value_counts().to_string()}")

## 4. Side-by-side review sheet

One row per building with every version of the answer next to each other: what the

earlier model predicted, what the validator did to it, the resulting truth, and

what the rule engine said — so a disagreement can be judged from the evidence in

the same row.

In [ ]:
comparison = build_comparison(val, preds)

comparison.to_csv(arm_comparison(ARM), index=False, encoding='utf-8')

print(f'wrote {arm_comparison(ARM).name}  ({len(comparison):,} rows)')

comparison.drop(columns='sentence').head(8)